# PrivateSQL + TPC-H Evaluation (Supplier Policy)

This notebook evaluates PrivateSQL-style differential privacy on TPC-H queries using the supplier policy. It builds Spark tables from raw TPC-H .tbl files, constructs protected and public session views, runs DP query variants (Q1, Q4, Q16), and computes error statistics across multiple DP runs.

In [1]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
os.environ["SPARK_HOME"] = "/Users/kinchintong/project/tumult-labs/spark-4.0.1-bin-hadoop3"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import findspark
findspark.init(os.environ["SPARK_HOME"])
from math import floor
from pyspark import SparkFiles
from pyspark.sql import SparkSession, types as T, functions as F
from tmlt.analytics.privacy_budget import PureDPBudget
from tmlt.analytics.protected_change import (AddOneRow, AddMaxRows, AddRowsWithID)
from tmlt.analytics.query_builder import QueryBuilder
from tmlt.analytics.session import Session, ColumnType
from tmlt.analytics.keyset import KeySet
from functools import reduce
from tmlt.analytics import (
    AddOneRow,
    PureDPBudget,
    QueryBuilder,
    Session,
    AddRowsWithID,
    GroupbyCountQuery,
)

from tmlt.analytics.truncation_strategy import TruncationStrategy

# Silence some Spark warnings
import warnings
warnings.simplefilter('ignore', UserWarning)
warnings.simplefilter('ignore', ResourceWarning)

findspark.init()

# Set up Spark
spark = (
    SparkSession.builder
    .appName("tpch-setup")
    .master("local[*]")           # drop if connecting to a cluster
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.driver.memory", "16g")
    .config("spark.executor.memory", "16g")
    .config("spark.driver.maxResultSize", "2g")
    .getOrCreate()
)
print("Spark version:", spark.version)

spark.sparkContext.setLogLevel("ERROR")

spark.conf.set("spark.sql.session.timeZone", "UTC")

# CREATE EMPTY TABLES (schema only) ----
def create_empty_tables():
    for t in TABLES:
        empty = spark.createDataFrame([], SCHEMAS[t])
        # managed Parquet tables; change to .format("delta") if using Delta
        #empty.write.mode("overwrite").format("parquet").saveAsTable(t)
    print(f"Created empty schema in database `{DB}`.")

# LOAD FROM CSV / .tbl (pipe-delimited with trailing '|') ----
def load_from_csv(src_dir: str):
    for t in TABLES:
        schema = SCHEMAS[t]
        path = f"{src_dir}/{t}.tbl"   # change to .csv if needed
        #only if the path exists
        if not os.path.exists(path):
            print(f"Path does not exist: {path}")
            continue
        
        print(f"Loading {t} from {path}...")

        # create a temporary wide schema of string columns (named) to absorb trailing '|'
        n_cols = len(schema.fields) + 1
        tmp_fields = [T.StructField(f"_c{i}", T.StringType(), True) for i in range(n_cols)]
        tmp_schema = T.StructType(tmp_fields)
        #print("tmp_schema:", tmp_schema)
        df_raw = (spark.read
                  .option("sep", "|")
                  .option("header", "false")
                  .option("mode", "PERMISSIVE")
                  .schema(tmp_schema)  # read wide to absorb trailing '|'
                  .csv(path))
        #print(f"Raw schema for {t}:")
        #df_raw.printSchema()
        
        # keep only first N columns and cast to target schema
        cols = df_raw.columns[:len(schema.fields)]
        df = df_raw.select([F.col(c).alias(schema.fields[i].name) for i, c in enumerate(cols)])
        # cast columns to target types
        for f in schema.fields:
            if isinstance(f.dataType, T.DateType):
                df = df.withColumn(f.name, F.to_date(F.col(f.name), "yyyy-MM-dd"))
            elif isinstance(f.dataType, T.DecimalType):
                df = df.withColumn(f.name, F.col(f.name).cast(f.dataType))
            elif isinstance(f.dataType, T.IntegerType):
                df = df.withColumn(f.name, F.col(f.name).cast("int"))
            # strings need no cast
        df.write.mode("overwrite").format("parquet").saveAsTable(t)
        print(f"Loaded {t} from {path}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/11 20:59:47 WARN Utils: Your hostname, kinchins-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.4.184 instead (on interface en0)
26/01/11 20:59:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/11 20:59:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/11 20:59:47 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/01/11 20:59:47 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/01/11 20:59:47 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Spark version: 4.0.1


In [2]:
DB = "tpch"                          # change as you like
OUTPUT="supplier_policy_session_1_eps_protected_change_mf_results"  # output directory for results

In [3]:
# Define canonical TPC-H schemas ----
dec = lambda p, s: T.DecimalType(p, s)
SCHEMAS = {
    "region":   T.StructType([
        T.StructField("r_regionkey", T.IntegerType(), False),
        T.StructField("r_name",      T.StringType(),  False),
        T.StructField("r_comment",   T.StringType(),  True),
    ]),
    "nation":   T.StructType([
        T.StructField("n_nationkey", T.IntegerType(), False),
        T.StructField("n_name",      T.StringType(),  False),
        T.StructField("n_regionkey", T.IntegerType(), False),
        T.StructField("n_comment",   T.StringType(),  True),
    ]),
    "part":     T.StructType([
        T.StructField("p_partkey",     T.IntegerType(), False),
        T.StructField("p_name",        T.StringType(),  False),
        T.StructField("p_mfgr",        T.StringType(),  False),
        T.StructField("p_brand",       T.StringType(),  False),
        T.StructField("p_type",        T.StringType(),  False),
        T.StructField("p_size",        T.IntegerType(), False),
        T.StructField("p_container",   T.StringType(),  False),
        T.StructField("p_retailprice", T.FloatType(),   False),
        T.StructField("p_comment",     T.StringType(),  True),
    ]),
    "supplier": T.StructType([
        T.StructField("s_suppkey",   T.IntegerType(), False),
        T.StructField("s_name",      T.StringType(),  False),
        T.StructField("s_address",   T.StringType(),  False),
        T.StructField("s_nationkey", T.IntegerType(), False),
        T.StructField("s_phone",     T.StringType(),  False),
        T.StructField("s_acctbal",   T.FloatType(),   False),
        T.StructField("s_comment",   T.StringType(),  True),
    ]),
    "partsupp": T.StructType([
        T.StructField("ps_partkey",    T.IntegerType(), False),
        T.StructField("ps_suppkey",    T.IntegerType(), False),
        T.StructField("ps_availqty",   T.IntegerType(), False),
        T.StructField("ps_supplycost", T.FloatType(),   False),
        T.StructField("ps_comment",    T.StringType(),  True),
    ]),
    "customer": T.StructType([
        T.StructField("c_custkey",   T.IntegerType(), False),
        T.StructField("c_name",      T.StringType(),  False),
        T.StructField("c_address",   T.StringType(),  False),
        T.StructField("c_nationkey", T.IntegerType(), False),
        T.StructField("c_phone",     T.StringType(),  False),
        T.StructField("c_acctbal",   T.FloatType(),   False),
        T.StructField("c_mktsegment",T.StringType(),  True),
        T.StructField("c_comment",   T.StringType(),  True),
    ]),
    "orders":   T.StructType([
        T.StructField("o_orderkey",     T.IntegerType(), False),
        T.StructField("o_custkey",      T.IntegerType(), False),
        T.StructField("o_orderstatus",  T.StringType(),  False),
        T.StructField("o_totalprice",   T.FloatType(),   False),
        T.StructField("o_orderdate",    T.DateType(),    False),
        T.StructField("o_orderpriority",T.StringType(),  False),
        T.StructField("o_clerk",        T.StringType(),  False),
        T.StructField("o_shippriority", T.IntegerType(), False),
        T.StructField("o_comment",      T.StringType(),  True),
    ]),
    "lineitem": T.StructType([
        T.StructField("l_orderkey",      T.IntegerType(), False),
        T.StructField("l_partkey",       T.IntegerType(), False),
        T.StructField("l_suppkey",       T.IntegerType(), False),
        T.StructField("l_linenumber",    T.IntegerType(), False),
        T.StructField("l_quantity",      T.FloatType(),   False),
        T.StructField("l_extendedprice", T.FloatType(),   False),
        T.StructField("l_discount",      T.FloatType(),   False),
        T.StructField("l_tax",           T.FloatType(),   False),
        T.StructField("l_returnflag",    T.StringType(),  False),
        T.StructField("l_linestatus",    T.StringType(),  False),
        T.StructField("l_shipdate",      T.DateType(),    False),
        T.StructField("l_commitdate",    T.DateType(),    False),
        T.StructField("l_receiptdate",   T.DateType(),    False),
        T.StructField("l_shipinstruct",  T.StringType(),  False),
        T.StructField("l_shipmode",      T.StringType(),  False),
        T.StructField("l_comment",       T.StringType(),  True),
    ]),
}
TABLES = ["region","nation","part","supplier","partsupp","customer","orders","lineitem"]
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB}")
spark.catalog.setCurrentDatabase(DB)

create_empty_tables()


Created empty schema in database `tpch`.


In [4]:
SRC_CSV_DIR = "./raw"    # where *.tbl live
load_from_csv(SRC_CSV_DIR)

# Quick smoke test:
spark.sql("SHOW TABLES").show(truncate=False)
# show record count for each table
for t in TABLES:
    print(f"Table {t} has {spark.sql(f'SELECT count(*) FROM {t}').collect()[0][0]} records.")

Loading region from ./raw/region.tbl...


Loaded region from ./raw/region.tbl
Loading nation from ./raw/nation.tbl...
Loaded nation from ./raw/nation.tbl
Loading part from ./raw/part.tbl...


Loaded part from ./raw/part.tbl
Loading supplier from ./raw/supplier.tbl...
Loaded supplier from ./raw/supplier.tbl
Loading partsupp from ./raw/partsupp.tbl...


Loaded partsupp from ./raw/partsupp.tbl
Loading customer from ./raw/customer.tbl...
Loaded customer from ./raw/customer.tbl
Loading orders from ./raw/orders.tbl...


Loaded orders from ./raw/orders.tbl
Loading lineitem from ./raw/lineitem.tbl...


Loaded lineitem from ./raw/lineitem.tbl
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|tpch     |customer |false      |
|tpch     |lineitem |false      |
|tpch     |nation   |false      |
|tpch     |orders   |false      |
|tpch     |part     |false      |
|tpch     |partsupp |false      |
|tpch     |region   |false      |
|tpch     |supplier |false      |
+---------+---------+-----------+

Table region has 5 records.
Table nation has 25 records.
Table part has 200000 records.
Table supplier has 10000 records.
Table partsupp has 800000 records.
Table customer has 150000 records.
Table orders has 1500000 records.
Table lineitem has 6001215 records.


In [5]:
from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.functions as F


def max_group_count(
    spark: SparkSession,
    table: str,
    key_col: str,
    where: str | None = None,
) -> tuple[str, int]:
    """
    Run the Spark SQL equivalent of:

        SELECT COUNT(key) AS count
        FROM table
        [WHERE ...]
        GROUP BY key
        ORDER BY count DESC
        LIMIT 1

    Returns: (key_value_as_string, count_as_int)
    """
    where_clause = f" WHERE {where} " if where else ""
    query = f"""
        SELECT {key_col} AS key, COUNT({key_col}) AS count
        FROM {table}
        {where_clause}
        GROUP BY {key_col}
        ORDER BY count DESC
        LIMIT 1
    """
    row = spark.sql(query).collect()
    if not row:
        raise ValueError("Query returned no rows (empty table or all rows filtered out).")

    print("Table:", table, ", key_col:", key_col, ", max count:", row[0]["count"])

    # row[0] has columns: key, count
    return str(row[0]["key"]), int(row[0]["count"])


max_group_count(spark, table="orders", key_col="o_custkey")
max_group_count(spark, table="lineitem", key_col="l_orderkey")
max_group_count(spark, table="lineitem", key_col="l_partkey")
max_group_count(spark, table="lineitem", key_col="l_suppkey")
max_group_count(spark, table="customer",key_col="c_nationkey")
max_group_count(spark, table="supplier",key_col="s_nationkey")
max_group_count(spark, table="partsupp",key_col="ps_partkey")
max_group_count(spark, table="partsupp",key_col="ps_suppkey")
max_group_count(spark, table="nation",key_col="n_regionkey")


Table: orders , key_col: o_custkey , max count: 41


Table: lineitem , key_col: l_orderkey , max count: 7
Table: lineitem , key_col: l_partkey , max count: 57
Table: lineitem , key_col: l_suppkey , max count: 694
Table: customer , key_col: c_nationkey , max count: 6161
Table: supplier , key_col: s_nationkey , max count: 438
Table: partsupp , key_col: ps_partkey , max count: 4
Table: partsupp , key_col: ps_suppkey , max count: 80
Table: nation , key_col: n_regionkey , max count: 5


('1', 5)

In [6]:
lineitem_mf = max_group_count(spark, table="lineitem", key_col="l_suppkey")[1]
print("Lineitem max frequency (mf) per supplier:", lineitem_mf)

partsupp_mf = max_group_count(spark, table="partsupp", key_col="ps_suppkey")[1]
print("Partsupp max frequency (mf) per supplier:", partsupp_mf)

supplier_mf = 1
print("Supplier max frequency (mf) per supplier:", supplier_mf)

Table: lineitem , key_col: l_suppkey , max count: 694
Lineitem max frequency (mf) per supplier: 694
Table: partsupp , key_col: ps_suppkey , max count: 80
Partsupp max frequency (mf) per supplier: 80
Supplier max frequency (mf) per supplier: 1


***
## TPC-H Queries Evaluated
- **Q13 - Customer Distribution Query**<br>
    This query seeks relationships between customers and the size of their orders.

- **Q16 - Parts/Supplier Relationship Query**<br>
    This query finds out how many suppliers can supply parts with given attributes. It might be used, for example, to
determine whether there is a sufficient number of suppliers for heavily ordered parts.

## PrivateSQL paper evaluation
### Dataset: TPC-H
### Privacy Policy: Supplier
### Privacy Budget ϵ: 1.0

In [15]:
SRC_CSV_DIR = "./raw"
dataframes = {}


TABLES = ["region","nation","part","supplier","partsupp","customer","orders","lineitem"]
for table_name in TABLES:
    path = f"{SRC_CSV_DIR}/{table_name}.tbl"
    schema = SCHEMAS[table_name]
    dataframes[table_name] = (spark.read
                .option("sep", "|")
                .option("header", "false")
                .option("mode", "PERMISSIVE")
                .schema(schema)
                .csv(path))

#needed for q1
lineitem_with_disc_view = (dataframes["lineitem"]
    .withColumn("disc_price",
                (dataframes["lineitem"]["l_extendedprice"].cast("double") * (1.0 - dataframes["lineitem"]["l_discount"].cast("double"))))
    .withColumn("charge",
                (dataframes["lineitem"]["l_extendedprice"].cast("double") * (1.0 - dataframes["lineitem"]["l_discount"].cast("double")) * (1.0 + dataframes["lineitem"]["l_tax"].cast("double"))))

)
dataframes["lineitem_with_disc"] = lineitem_with_disc_view

WORD1 = 'special'
WORD2 = 'requests'
q13_view = f"""
SELECT
  c.c_custkey,
  COUNT(o.o_orderkey) AS c_count
FROM customer c
LEFT JOIN orders o
  ON  o.o_custkey = c.c_custkey
  AND o_comment NOT LIKE '%{WORD1}%{WORD2}%'
GROUP BY c.c_custkey;
"""

V_q13 = spark.sql(q13_view)
dataframes["V_q13"] = V_q13


for table_name in dataframes:
    print(f"Schema for table: {table_name}")
    dataframes[table_name].show(5)

def initialize_session():
    #budget = PureDPBudget(float("inf"))
    budget = PureDPBudget(1)

    id_space = "supplier_policy_id_space"

    session = (
        Session.Builder()
        .with_privacy_budget(budget)
        .with_id_space(id_space)
        .with_private_dataframe(
            "lineitem_with_disc",
            dataframes["lineitem_with_disc"],
            protected_change=AddMaxRows(lineitem_mf),
        )
        .with_private_dataframe(
            "lineitem",
            dataframes["lineitem"],
            protected_change=AddMaxRows(lineitem_mf),
        )
        .with_public_dataframe(
            "orders",
            dataframes["orders"],
        )
        .with_public_dataframe(
            "customer",
            dataframes["customer"],
        )
        .with_public_dataframe(
            "region",
            dataframes["region"],
        )
        .with_public_dataframe(
            "nation",
            dataframes["nation"],
        )
        .with_public_dataframe(
            "part",
            dataframes["part"],
        )
        .with_public_dataframe(
            "V_q13",
            dataframes["V_q13"],
        )
        .with_private_dataframe(
            "supplier",
            dataframes["supplier"],
            protected_change=AddMaxRows(supplier_mf),
        )
        .with_private_dataframe(
            "partsupp",
            dataframes["partsupp"],
            protected_change=AddMaxRows(partsupp_mf),
        )
        .build()
    )
    return session

Schema for table: region
+-----------+-----------+--------------------+
|r_regionkey|     r_name|           r_comment|
+-----------+-----------+--------------------+
|          0|     AFRICA|lar deposits. bli...|
|          1|    AMERICA|hs use ironic, ev...|
|          2|       ASIA|ges. thinly even ...|
|          3|     EUROPE|ly final courts c...|
|          4|MIDDLE EAST|uickly special ac...|
+-----------+-----------+--------------------+

Schema for table: nation
+-----------+---------+-----------+--------------------+
|n_nationkey|   n_name|n_regionkey|           n_comment|
+-----------+---------+-----------+--------------------+
|          0|  ALGERIA|          0| haggle. carefull...|
|          1|ARGENTINA|          1|al foxes promise ...|
|          2|   BRAZIL|          1|y alongside of th...|
|          3|   CANADA|          1|eas hang ironic, ...|
|          4|    EGYPT|          4|y above the caref...|
+-----------+---------+-----------+--------------------+
only showing 

In [8]:
def query_dp_eval(eps: PureDPBudget, query_num, query_dp_run, num_runs: int = 1, start_run: int = 0):
  for run in range(start_run,num_runs):
    print(f"Running Q{query_num} DP run {run+1}/{num_runs} with epsilon={eps.epsilon}...")

    session = initialize_session()
    runx = query_dp_run(eps, session)
    
    output_file = f"./output/{OUTPUT}/q{query_num}/q{query_num}_dp_result_eps_{eps.epsilon}_runs_{run+1}_csv"
    print(output_file)
    runx.coalesce(1).write \
      .mode("overwrite") \
      .option("header", "true") \
      .csv(output_file)

In [9]:
def q1_dp_run(eps: PureDPBudget, session: Session):
  
  #1) Create V_q1 with casts + derived columns
  DELTA = 90

  q1_view = (
    QueryBuilder("lineitem_with_disc")
      .select([
          "l_returnflag",
          "l_linestatus",
          "l_quantity",
          "l_extendedprice",
          "l_discount",
          "l_tax",
          "l_shipdate",
          "disc_price",
          "charge",
      ])
  )

  # delete V_q1 only if it exists to avoid errors
  try:
    session.delete_view("V_q1")
  except Exception as e:
    pass

  session.create_view(source_id="V_q1", query_expr=q1_view, cache=True)

  #2) Group + filter (use Spark-friendly interval)
  groupby_keys = KeySet.from_dict({
      "l_returnflag": ["A", "N", "R"],
      "l_linestatus": ["F", "O"],
  })

  q1_linear_grouped = (
      QueryBuilder("V_q1")
        .filter(f"l_shipdate <= DATE '1998-12-01' - INTERVAL {DELTA} DAYS")
        .groupby(groupby_keys)
        .count(name="count_order")
  )

  dp_count_order    = session.evaluate(q1_linear_grouped,    privacy_budget=eps)
  return dp_count_order

    

In [10]:
def q1_no_view_dp_run(eps: PureDPBudget, session: Session):
  
  #1) Create V_q1 with casts + derived columns
  DELTA = 90

  groupby_keys = KeySet.from_dict({
      "l_returnflag": ["A", "N", "R"],
      "l_linestatus": ["F", "O"],
  })

  q1 = (
    QueryBuilder("lineitem_with_disc")
      .select([
          "l_returnflag",
          "l_linestatus",
          "l_quantity",
          "l_extendedprice",
          "l_discount",
          "l_tax",
          "l_shipdate",
          "disc_price",
          "charge",
      ]).filter(f"l_shipdate <= DATE '1998-12-01' - INTERVAL {DELTA} DAYS")
        .groupby(groupby_keys)
        .count(name="count_order")
  )

  dp_count_order    = session.evaluate(q1,    privacy_budget=eps)
  return dp_count_order

    

In [11]:
def q4_dp_run(eps: PureDPBudget, session: Session):
    print("Building Q4 deduped view V_q4...")
    from tmlt.analytics.query_builder import QueryBuilder

    q_lineitem_filtered = (
        QueryBuilder("lineitem")
        .filter("l_commitdate < l_receiptdate")
        .select(["l_orderkey"])
        .rename({"l_orderkey": "orderkey"})
    )

    # orders columns (include orderdate so you DON'T need to join orders again later)
    q_orders = (
        QueryBuilder("orders")
        .select(["o_orderkey", "o_orderpriority", "o_orderdate"])
        .rename({"o_orderkey": "orderkey"})
    )

    orders = dataframes["orders"].withColumnRenamed("o_orderkey", "orderkey")

    # join
    start_date = "1993-07-01"
    q4_view = q_lineitem_filtered.join_public(
        orders,
        join_columns=["orderkey"],
    ).select(["orderkey", "o_orderpriority", "o_orderdate"]
    ).filter(
        f"o_orderdate >= DATE '{start_date}' AND "
        f"o_orderdate <  DATE '{start_date}' + INTERVAL 3 MONTH"
    )

    # create session view
    try:
        session.delete_view("V_q4_1")
    except Exception:
        pass

    session.create_view(source_id="V_q4_1", query_expr=q4_view, cache=True)


    eps = PureDPBudget(epsilon=1.0)

    groupby_keys = KeySet.from_dict({
        "o_orderpriority": ["1-URGENT", "2-HIGH", "3-MEDIUM", "4-NOT SPECIFIED", "5-LOW"]
    })


    q4_1 = (
        QueryBuilder("V_q4_1")
        .select(["o_orderpriority"])
        .groupby(groupby_keys)
    )

    q4_count = q4_1.count(name="order_count")

    q4_dp_result = session.evaluate(q4_count, privacy_budget=eps)

    return q4_dp_result

In [12]:
def q16_dp_DropExcess_1_run(eps: PureDPBudget, session: Session):
    from pyspark.sql.functions import col
    q16_view_part = (
        QueryBuilder("part")
        .select(["p_partkey", "p_brand", "p_type", "p_size"])
        .rename({"p_partkey": "partkey"})
    )

    q16_view_partsupp = (
        QueryBuilder("partsupp")
        .select(["ps_partkey", "ps_suppkey"])
        .rename({"ps_partkey": "partkey", "ps_suppkey": "suppkey"}) 
    )

    q16_view_supplier = (
        QueryBuilder("supplier")
        .select(["s_suppkey", "s_comment"])
        .filter("s_comment NOT LIKE '%Customer%Complaints%'")
        .rename({"s_suppkey": "suppkey"})
    )

    q_16_view_1 = q16_view_partsupp.join_private(
        right_operand=q16_view_supplier,
        join_columns=["suppkey"],
        truncation_strategy_left=TruncationStrategy.DropExcess(1),
        truncation_strategy_right=TruncationStrategy.DropExcess(1),
    ) 

    part = dataframes["part"].select(
        "p_brand",
        "p_type",
        "p_size",
        col("p_partkey").alias("partkey"),
        )

    q_16_view = (
        q_16_view_1.join_public(
            part,
            join_columns=["partkey"]
        )
    )

    try:
        session.delete_view("V_q16")
    except Exception:
        pass

    session.create_view(source_id="V_q16", query_expr=q_16_view, cache=True)

    '''
    SELECT
        p_brand, p_type, p_size,
        COUNT(*) AS supplier_cnt
    FROM V_q16
    WHERE p_brand <> '{BRAND}'
        AND p_type NOT LIKE '{TYPE_PREFIX}%'
        AND p_size IN ({sizes_csv})
    GROUP BY p_brand, p_type, p_size
    '''

    syllable1 = ["STANDARD", "SMALL", "MEDIUM", "LARGE", "ECONOMY", "PROMO"]
    syllable2 = ["ANODIZED", "BURNISHED", "PLATED", "POLISHED", "BRUSHED"]
    syllable3 = ["TIN", "NICKEL", "BRASS", "STEEL", "COPPER"]

    TPC_H_TYPES = [
        f"{s1} {s2} {s3}"
        for s1 in syllable1
        for s2 in syllable2
        for s3 in syllable3
    ]

    groupby_keys = KeySet.from_dict({
        "p_brand": ["Brand#11", "Brand#12", "Brand#13","Brand#14", "Brand#15", 
                    "Brand#21", "Brand#22", "Brand#23","Brand#24", "Brand#25",
                    "Brand#31", "Brand#32", "Brand#33","Brand#34", "Brand#35",
                    "Brand#41", "Brand#42", "Brand#43","Brand#44", "Brand#45",
                    "Brand#51", "Brand#52", "Brand#53","Brand#54", "Brand#55"],
        "p_type": [
                        f"{s1} {s2} {s3}"
                        for s1 in syllable1
                        for s2 in syllable2
                        for s3 in syllable3
                    ],
        "p_size": list(range(1, 51)),
    })

    BRAND = "Brand#12"
    TYPE_PREFIX = "LARGE ANODIZED"
    SIZES = [3, 6, 9, 12, 18, 24, 36, 48]
    sizes_csv = ",".join(map(str, SIZES))

    q16_filtered = (
        QueryBuilder("V_q16")
            .filter(f"p_brand <> '{BRAND}' AND p_type NOT LIKE '{TYPE_PREFIX}%' AND p_size IN ({sizes_csv})")
            .select(["p_brand", "p_type", "p_size"])
            .groupby(groupby_keys)
            .count(name="supplier_cnt")
    )

    dp_result = session.evaluate(q16_filtered, privacy_budget=eps)
    return dp_result


In [13]:
def q16_dp_DropExcess_mf_run(eps: PureDPBudget, session: Session):
    from pyspark.sql.functions import col
    q16_view_part = (
        QueryBuilder("part")
        .select(["p_partkey", "p_brand", "p_type", "p_size"])
        .rename({"p_partkey": "partkey"})
    )

    q16_view_partsupp = (
        QueryBuilder("partsupp")
        .select(["ps_partkey", "ps_suppkey"])
        .rename({"ps_partkey": "partkey", "ps_suppkey": "suppkey"}) 
    )

    q16_view_supplier = (
        QueryBuilder("supplier")
        .select(["s_suppkey", "s_comment"])
        .filter("s_comment NOT LIKE '%Customer%Complaints%'")
        .rename({"s_suppkey": "suppkey"})
    )

    q_16_view_1 = q16_view_partsupp.join_private(
        right_operand=q16_view_supplier,
        join_columns=["suppkey"],
        truncation_strategy_left=TruncationStrategy.DropExcess(partsupp_mf),
        truncation_strategy_right=TruncationStrategy.DropExcess(supplier_mf),
    ) 

    part = dataframes["part"].select(
        "p_brand",
        "p_type",
        "p_size",
        col("p_partkey").alias("partkey"),
        )

    q_16_view = (
        q_16_view_1.join_public(
            part,
            join_columns=["partkey"]
        )
    )

    try:
        session.delete_view("V_q16")
    except Exception:
        pass

    session.create_view(source_id="V_q16", query_expr=q_16_view, cache=True)

    '''
    SELECT
        p_brand, p_type, p_size,
        COUNT(*) AS supplier_cnt
    FROM V_q16
    WHERE p_brand <> '{BRAND}'
        AND p_type NOT LIKE '{TYPE_PREFIX}%'
        AND p_size IN ({sizes_csv})
    GROUP BY p_brand, p_type, p_size
    '''

    syllable1 = ["STANDARD", "SMALL", "MEDIUM", "LARGE", "ECONOMY", "PROMO"]
    syllable2 = ["ANODIZED", "BURNISHED", "PLATED", "POLISHED", "BRUSHED"]
    syllable3 = ["TIN", "NICKEL", "BRASS", "STEEL", "COPPER"]

    TPC_H_TYPES = [
        f"{s1} {s2} {s3}"
        for s1 in syllable1
        for s2 in syllable2
        for s3 in syllable3
    ]

    groupby_keys = KeySet.from_dict({
        "p_brand": ["Brand#11", "Brand#12", "Brand#13","Brand#14", "Brand#15", 
                    "Brand#21", "Brand#22", "Brand#23","Brand#24", "Brand#25",
                    "Brand#31", "Brand#32", "Brand#33","Brand#34", "Brand#35",
                    "Brand#41", "Brand#42", "Brand#43","Brand#44", "Brand#45",
                    "Brand#51", "Brand#52", "Brand#53","Brand#54", "Brand#55"],
        "p_type": [
                        f"{s1} {s2} {s3}"
                        for s1 in syllable1
                        for s2 in syllable2
                        for s3 in syllable3
                    ],
        "p_size": list(range(1, 51)),
    })

    BRAND = "Brand#12"
    TYPE_PREFIX = "LARGE ANODIZED"
    SIZES = [3, 6, 9, 12, 18, 24, 36, 48]
    sizes_csv = ",".join(map(str, SIZES))

    q16_filtered = (
        QueryBuilder("V_q16")
            .filter(f"p_brand <> '{BRAND}' AND p_type NOT LIKE '{TYPE_PREFIX}%' AND p_size IN ({sizes_csv})")
            .select(["p_brand", "p_type", "p_size"])
            .groupby(groupby_keys)
            .count(name="supplier_cnt")
    )

    dp_result = session.evaluate(q16_filtered, privacy_budget=eps)
    return dp_result


In [16]:
for run in [50]:
    eps = PureDPBudget(1)
    #query_dp_eval(eps, query_num="1", query_dp_run=q1_dp_run, num_runs=run)
    query_dp_eval(eps, query_num="1_no_view", query_dp_run=q1_no_view_dp_run, num_runs=run, start_run=10)
    query_dp_eval(eps, query_num="4", query_dp_run=q4_dp_run, num_runs=run, start_run=10)   

    query_dp_eval(eps, query_num="16_DropExcess_1", query_dp_run=q16_dp_DropExcess_1_run, num_runs=run, start_run=10)
    query_dp_eval(eps, query_num="16_DropExcess_mf", query_dp_run=q16_dp_DropExcess_mf_run, num_runs=run, start_run=10)
    

Running Q1_no_view DP run 11/50 with epsilon=1...
 QueryExprCompiler: output_measure =  PureDP()
>>> Query after rewrite:  GroupByCount(child=Filter(child=Select(child=PrivateSource(source_id='lineitem_with_disc'), columns=('l_returnflag', 'l_linestatus', 'l_quantity', 'l_extendedprice', 'l_discount', 'l_tax', 'l_shipdate', 'disc_price', 'charge')), condition="l_shipdate <= DATE '1998-12-01' - INTERVAL 90 DAYS"), groupby_keys=<tmlt.analytics.keyset._keyset.KeySet object at 0x171bb55e0>, output_column='count_order', mechanism=<CountMechanism.DEFAULT: 1>, core_mechanism=<NoiseMechanism.GEOMETRIC: 2>)
>>> Stability: 
   NamedTable(lineitem_with_disc)  =  694
   NamedTable(lineitem)  =  694
   NamedTable(supplier)  =  1
   NamedTable(partsupp)  =  80
>>> GroupByCount - accept
>>> BaseMeasurementVisitor.visit_groupby_count
>>> BaseMeasurementVisitor.visit_groupby_count: mid_stability= 694
>>> BaseMeasurementVisitor._build_adaptive_groupby_agg_and_noise_info stability= 694
>>> BaseMeasuremen

./output/supplier_policy_session_1_eps_protected_change_mf_results/q1_no_view/q1_no_view_dp_result_eps_1_runs_11_csv
Running Q1_no_view DP run 12/50 with epsilon=1...
 QueryExprCompiler: output_measure =  PureDP()
>>> Query after rewrite:  GroupByCount(child=Filter(child=Select(child=PrivateSource(source_id='lineitem_with_disc'), columns=('l_returnflag', 'l_linestatus', 'l_quantity', 'l_extendedprice', 'l_discount', 'l_tax', 'l_shipdate', 'disc_price', 'charge')), condition="l_shipdate <= DATE '1998-12-01' - INTERVAL 90 DAYS"), groupby_keys=<tmlt.analytics.keyset._keyset.KeySet object at 0x168dc3650>, output_column='count_order', mechanism=<CountMechanism.DEFAULT: 1>, core_mechanism=<NoiseMechanism.GEOMETRIC: 2>)
>>> Stability: 
   NamedTable(lineitem_with_disc)  =  694
   NamedTable(lineitem)  =  694
   NamedTable(supplier)  =  1
   NamedTable(partsupp)  =  80
>>> GroupByCount - accept
>>> BaseMeasurementVisitor.visit_groupby_count
>>> BaseMeasurementVisitor.visit_groupby_count: mid_s

./output/supplier_policy_session_1_eps_protected_change_mf_results/q1_no_view/q1_no_view_dp_result_eps_1_runs_12_csv
Running Q1_no_view DP run 13/50 with epsilon=1...
 QueryExprCompiler: output_measure =  PureDP()
>>> Query after rewrite:  GroupByCount(child=Filter(child=Select(child=PrivateSource(source_id='lineitem_with_disc'), columns=('l_returnflag', 'l_linestatus', 'l_quantity', 'l_extendedprice', 'l_discount', 'l_tax', 'l_shipdate', 'disc_price', 'charge')), condition="l_shipdate <= DATE '1998-12-01' - INTERVAL 90 DAYS"), groupby_keys=<tmlt.analytics.keyset._keyset.KeySet object at 0x1727bf8c0>, output_column='count_order', mechanism=<CountMechanism.DEFAULT: 1>, core_mechanism=<NoiseMechanism.GEOMETRIC: 2>)
>>> Stability: 
   NamedTable(lineitem_with_disc)  =  694
   NamedTable(lineitem)  =  694
   NamedTable(supplier)  =  1
   NamedTable(partsupp)  =  80
>>> GroupByCount - accept
>>> BaseMeasurementVisitor.visit_groupby_count
>>> BaseMeasurementVisitor.visit_groupby_count: mid_s

./output/supplier_policy_session_1_eps_protected_change_mf_results/q1_no_view/q1_no_view_dp_result_eps_1_runs_13_csv
Running Q1_no_view DP run 14/50 with epsilon=1...
 QueryExprCompiler: output_measure =  PureDP()
>>> Query after rewrite:  GroupByCount(child=Filter(child=Select(child=PrivateSource(source_id='lineitem_with_disc'), columns=('l_returnflag', 'l_linestatus', 'l_quantity', 'l_extendedprice', 'l_discount', 'l_tax', 'l_shipdate', 'disc_price', 'charge')), condition="l_shipdate <= DATE '1998-12-01' - INTERVAL 90 DAYS"), groupby_keys=<tmlt.analytics.keyset._keyset.KeySet object at 0x172dbae70>, output_column='count_order', mechanism=<CountMechanism.DEFAULT: 1>, core_mechanism=<NoiseMechanism.GEOMETRIC: 2>)
>>> Stability: 
   NamedTable(lineitem_with_disc)  =  694
   NamedTable(lineitem)  =  694
   NamedTable(supplier)  =  1
   NamedTable(partsupp)  =  80
>>> GroupByCount - accept
>>> BaseMeasurementVisitor.visit_groupby_count
>>> BaseMeasurementVisitor.visit_groupby_count: mid_s

./output/supplier_policy_session_1_eps_protected_change_mf_results/q1_no_view/q1_no_view_dp_result_eps_1_runs_14_csv
Running Q1_no_view DP run 15/50 with epsilon=1...
 QueryExprCompiler: output_measure =  PureDP()
>>> Query after rewrite:  GroupByCount(child=Filter(child=Select(child=PrivateSource(source_id='lineitem_with_disc'), columns=('l_returnflag', 'l_linestatus', 'l_quantity', 'l_extendedprice', 'l_discount', 'l_tax', 'l_shipdate', 'disc_price', 'charge')), condition="l_shipdate <= DATE '1998-12-01' - INTERVAL 90 DAYS"), groupby_keys=<tmlt.analytics.keyset._keyset.KeySet object at 0x171bb9b50>, output_column='count_order', mechanism=<CountMechanism.DEFAULT: 1>, core_mechanism=<NoiseMechanism.GEOMETRIC: 2>)
>>> Stability: 
   NamedTable(lineitem_with_disc)  =  694
   NamedTable(lineitem)  =  694
   NamedTable(supplier)  =  1
   NamedTable(partsupp)  =  80
>>> GroupByCount - accept
>>> BaseMeasurementVisitor.visit_groupby_count
>>> BaseMeasurementVisitor.visit_groupby_count: mid_s

...

./output/supplier_policy_session_1_eps_protected_change_mf_results/q16_DropExcess_mf/q16_DropExcess_mf_dp_result_eps_1_runs_46_csv
Running Q16_DropExcess_mf DP run 47/50 with epsilon=1...
 QueryExprCompiler: output_measure =  PureDP()
create_view accountant= <tmlt.core.measurements.interactive_measurements.PrivacyAccountant object at 0x172de7b90>
create_view transformation= <tmlt.core.transformations.chaining.ChainTT object at 0x16a0ea870>
create_view constraints= []
create_view ref = TableReference(TemporaryTable(4567439680))
create_view accountant= <tmlt.core.measurements.interactive_measurements.PrivacyAccountant object at 0x172de7b90>
create_view accountant.d_in= {NamedTable(name='lineitem_with_disc'): 694, NamedTable(name='lineitem'): 694, NamedTable(name='supplier'): 1, NamedTable(name='partsupp'): 80, TemporaryTable(6203095280): 80, TemporaryTable(6203138432): 80, TemporaryTable(6203083616): 1, TemporaryTable(6203092928): 1, TemporaryTable(6203453184): 1, TemporaryTable(62031302

./output/supplier_policy_session_1_eps_protected_change_mf_results/q16_DropExcess_mf/q16_DropExcess_mf_dp_result_eps_1_runs_47_csv
Running Q16_DropExcess_mf DP run 48/50 with epsilon=1...
 QueryExprCompiler: output_measure =  PureDP()
create_view accountant= <tmlt.core.measurements.interactive_measurements.PrivacyAccountant object at 0x172de6a50>
create_view transformation= <tmlt.core.transformations.chaining.ChainTT object at 0x171bb87a0>
create_view constraints= []
create_view ref = TableReference(TemporaryTable(6203072320))
create_view accountant= <tmlt.core.measurements.interactive_measurements.PrivacyAccountant object at 0x172de6a50>
create_view accountant.d_in= {NamedTable(name='lineitem_with_disc'): 694, NamedTable(name='lineitem'): 694, NamedTable(name='supplier'): 1, NamedTable(name='partsupp'): 80, TemporaryTable(6204078176): 80, TemporaryTable(6204067616): 80, TemporaryTable(6204078944): 1, TemporaryTable(6204066800): 1, TemporaryTable(6204064304): 1, TemporaryTable(62040772

./output/supplier_policy_session_1_eps_protected_change_mf_results/q16_DropExcess_mf/q16_DropExcess_mf_dp_result_eps_1_runs_48_csv
Running Q16_DropExcess_mf DP run 49/50 with epsilon=1...
 QueryExprCompiler: output_measure =  PureDP()
create_view accountant= <tmlt.core.measurements.interactive_measurements.PrivacyAccountant object at 0x171c107a0>
create_view transformation= <tmlt.core.transformations.chaining.ChainTT object at 0x171cabb30>
create_view constraints= []
create_view ref = TableReference(TemporaryTable(6208085008))
create_view accountant= <tmlt.core.measurements.interactive_measurements.PrivacyAccountant object at 0x171c107a0>
create_view accountant.d_in= {NamedTable(name='lineitem_with_disc'): 694, NamedTable(name='lineitem'): 694, NamedTable(name='supplier'): 1, NamedTable(name='partsupp'): 80, TemporaryTable(6203138912): 80, TemporaryTable(6203145680): 80, TemporaryTable(6203137712): 1, TemporaryTable(6203133728): 1, TemporaryTable(6203135168): 1, TemporaryTable(62031430

./output/supplier_policy_session_1_eps_protected_change_mf_results/q16_DropExcess_mf/q16_DropExcess_mf_dp_result_eps_1_runs_49_csv
Running Q16_DropExcess_mf DP run 50/50 with epsilon=1...
 QueryExprCompiler: output_measure =  PureDP()
create_view accountant= <tmlt.core.measurements.interactive_measurements.PrivacyAccountant object at 0x169176ff0>
create_view transformation= <tmlt.core.transformations.chaining.ChainTT object at 0x171bb7290>
create_view constraints= []
create_view ref = TableReference(TemporaryTable(5762514144))
create_view accountant= <tmlt.core.measurements.interactive_measurements.PrivacyAccountant object at 0x169176ff0>
create_view accountant.d_in= {NamedTable(name='lineitem_with_disc'): 694, NamedTable(name='lineitem'): 694, NamedTable(name='supplier'): 1, NamedTable(name='partsupp'): 80, TemporaryTable(6203145632): 80, TemporaryTable(6203446944): 80, TemporaryTable(6203132096): 1, TemporaryTable(6203138192): 1, TemporaryTable(6204069488): 1, TemporaryTable(62040752

./output/supplier_policy_session_1_eps_protected_change_mf_results/q16_DropExcess_mf/q16_DropExcess_mf_dp_result_eps_1_runs_50_csv


***
## Calcuate Error

In [ ]:
from typing import List
import matplotlib.pyplot as plt
import pandas as pd

def compute_errors_privatesql_style(
    dp_df,
    truth_df,
    keys: List[str],
    metric: str = "count_order",
    run_col: str = "run_id",
    join_type: str = "left",
    truth_missing_as_zero: bool = True,
):
    """
    Compute PrivateSQL-style errors per output group per run.
      Qerror(y, y~)   = |y - y~|
      RelError(y, y~) = |y - y~| / max(50, y)

    Returns columns:
      run_id,
      keys,
      <metric>_truth,
      <metric>_dp,
      qerror_<metric>,
      relerror_<metric>
    """

    # Prepare DP side
    dp = (
        dp_df
        .withColumn(metric, F.col(metric).cast("double"))
        .withColumnRenamed(metric, f"{metric}_dp")
    )
    #dp.show(5)

    # Prepare truth side
    tr = (
        truth_df
        .withColumn(metric, F.col(metric).cast("double"))
        .withColumnRenamed(metric, f"{metric}_truth")
    )

    #tr.show(5)
    
    joined = dp.alias("dp").join(tr.alias("tr"), on=keys, how=join_type)

    if truth_missing_as_zero:
        joined = joined.withColumn(
            f"{metric}_truth",
            F.coalesce(F.col(f"{metric}_truth"), F.lit(0.0))
        )

    # Absolute error |y - y~|
    joined = joined.withColumn(
        f"qerror",
        F.abs(F.col(f"{metric}_truth") - F.col(f"{metric}_dp"))
    )

    # Relative error |y - y~| / max(50, y)
    joined = joined.withColumn(
        f"relerror",
        F.col(f"qerror") /
        F.greatest(F.lit(50.0), F.col(f"{metric}_truth"))
    )

    return joined.select(
        run_col,
        *keys,
        f"{metric}_truth",
        f"{metric}_dp",
        f"qerror",
        f"relerror",
    )

tpch_query_results = {}

In [ ]:
def q16_calculate_errors(test_profile: str, test_case: str):

# -------------------------
# CONFIG
# -------------------------
    keys = ["p_brand","p_type","p_size"]
    metrics = "supplier_cnt"

    eps_val = 1
    truth_path = "./output/no_dp/q16_result_csv"
    # -------------------------
    # LOAD TRUTH
    # -------------------------
    truth = (spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(truth_path)
        .select(*keys, "supplier_cnt")
        .withColumn("supplier_cnt", F.col("supplier_cnt").cast("long"))
    )

    # -------------------------
    # LOAD RUNS (ALL 1000)
    # -------------------------
    runs_glob = f"./output/{test_profile}/{test_case}/{test_case}_dp_result_eps_{eps_val}_runs_*_csv"

    dp_runs = (spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(runs_glob)
        .withColumn(
            "run_id",
            F.regexp_extract(F.input_file_name(), r"runs_(\d+)_csv", 1).cast("int")
        )
        .select("run_id", *keys, "supplier_cnt")
        .withColumn("supplier_cnt", F.col("supplier_cnt").cast("long"))
    )
    print("Q16")
    print("Distinct runs =", dp_runs.select("run_id").distinct().count())

    err_df = compute_errors_privatesql_style(
        dp_df=dp_runs,
        truth_df=truth,
        keys=keys,
        metric=metrics,
        run_col="run_id",
        join_type="left",
        truth_missing_as_zero=True,
    )

    #err_df.show(truncate=False)

    per_run = err_df.groupBy("run_id").agg(
        F.avg("qerror").alias("avg_qerror_per_run"),
        F.avg("relerror").alias("avg_relerror_per_run"),
    ).orderBy("run_id")

    relerror = (
        per_run
        .select("avg_relerror_per_run")
        .dropna()
        .toPandas()
    )

    qerror = (
        per_run
        .select("avg_qerror_per_run")
        .dropna()
        .toPandas()
    )

    overall = per_run.agg(
        F.avg("avg_qerror_per_run").alias("mean_qerror_over_runs"),
        F.avg("avg_relerror_per_run").alias("mean_relerror_over_runs"),
    )

    key=f'{OUTPUT}_{test_case}'
    with open(
        f"./output/{OUTPUT}/{key}.csv",
        "w"
    ) as f:
        f.write(key)
        f.write(",")
        f.write(",".join(str(x) for x in relerror.iloc[:, 0]))

    result = {
            "mean_qerror_over_runs": float(overall.collect()[0]["mean_qerror_over_runs"]),
            "mean_relerror_over_runs": float(overall.collect()[0]["mean_relerror_over_runs"]),
            "qerror": qerror,
            "relerror": relerror
        }

    plt.figure(figsize=(4, 4))
    plt.boxplot(relerror, showfliers=False)
    plt.xticks([1], [test_case])
    plt.xlabel("TPC-H Query (Supplier policy)")
    plt.title("Relative Error for Q16 with OpenDP-Tumult")
    plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
    plt.tight_layout()
    plt.show()

    '''
    plt.figure(figsize=(4, 4))
    plt.boxplot(qerror, showfliers=False)
    plt.xticks([1], [test_case])
    plt.xlabel("TPC-H Query (Supplier policy)")
    plt.title("Absolute Error for Q16 with OpenDP-Tumult")
    plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
    plt.tight_layout()
    plt.show()
    '''
    return result

In [ ]:
x = q16_calculate_errors(OUTPUT, test_case="q16_DropExcess_1")
tpch_query_results["q16_DropExcess_1"] = x

In [ ]:
x = q16_calculate_errors(OUTPUT, test_case="q16_DropExcess_mf")
tpch_query_results["q16_DropExcess_mf"] = x

***


In [ ]:
print("opendp eval with TPCH and supplier policy Results Summary:")

import matplotlib.pyplot as plt

# Use a stable list of labels (convert to strings) and numeric positions for plotting
queries = list(tpch_query_results.keys())
queries_label = [str(q) for q in queries]
print("Queries evaluated:", queries_label)

avg_qerror = [tpch_query_results[q]["mean_qerror_over_runs"] for q in queries]
raw_qerror = [tpch_query_results[q]["qerror"] for q in queries]
print("Average Qerror per query:", avg_qerror)
avg_relerror = [tpch_query_results[q]["mean_relerror_over_runs"] for q in queries]
print("Average RelError per query:", avg_relerror)

positions = list(range(len(queries_label)))

# Absolute error bar plot
fig, ax = plt.subplots(figsize=(4, 4))
ax.bar(positions, avg_qerror, color='tab:blue')
ax.set_xticks(positions)
ax.set_xticklabels(queries_label, rotation=45, ha='right')
ax.set_xlabel("TPC-H Query (Supplier policy)")
ax.set_ylabel("Average Absolute Error (Qerror)")
ax.set_title("Average Absolute Error Across Runs")
plt.tight_layout()
plt.show()

# Relative error bar plot
fig2, ax2 = plt.subplots(figsize=(4, 4))
ax2.bar(positions, avg_relerror, color='tab:orange')
ax2.set_xticks(positions)
ax2.set_xticklabels(queries_label, rotation=45, ha='right')
ax2.set_xlabel("TPC-H Query (Supplier policy)")
ax2.set_ylabel("Average Relative Error")
ax2.set_title("Average Relative Error Across Runs")
plt.tight_layout()
plt.show()

raw_qerror = [tpch_query_results[q]["qerror"] for q in queries]
raw_relerror = [tpch_query_results[q]["relerror"] for q in queries]
import numpy as np
raw_qerror_1d = [np.asarray(x).ravel() for x in raw_qerror]

plt.figure(figsize=(4, 4))
plt.boxplot(
    raw_qerror_1d,
    labels=queries_label,
    showfliers=False
)
plt.xticks(rotation=45, ha='right')
plt.xlabel("TPC-H Query (Supplier policy)")
plt.ylabel("Average Absolute Error (Qerror)")
plt.title("qerror Distribution Across Runs")
plt.tight_layout()
plt.show()

raw_relerror_1d = [np.asarray(x).ravel() for x in raw_relerror]

plt.figure(figsize=(4, 4))
plt.boxplot(
    raw_relerror_1d,
    labels=queries_label,
    showfliers=False
)
plt.xticks(rotation=45, ha='right')
plt.xlabel("TPC-H Query (Supplier policy)")
plt.ylabel("Average Relative Error (RelError)")
plt.title("relerror Distribution Across Runs")
plt.tight_layout()
plt.show()